# Lab | Web Scraping

Welcome to the "Books to Scrape" Web Scraping Adventure Lab!

**Objective**

In this lab, we will embark on a mission to unearth valuable insights from the data available on Books to Scrape, an online platform showcasing a wide variety of books. As data analyst, you have been tasked with scraping a specific subset of book data from Books to Scrape to assist publishing companies in understanding the landscape of highly-rated books across different genres. Your insights will help shape future book marketing strategies and publishing decisions.

**Background**

In a world where data has become the new currency, businesses are leveraging big data to make informed decisions that drive success and profitability. The publishing industry, much like others, utilizes data analytics to understand market trends, reader preferences, and the performance of books based on factors such as genre, author, and ratings. Books to Scrape serves as a rich source of such data, offering detailed information about a diverse range of books, making it an ideal platform for extracting insights to aid in informed decision-making within the literary world.

**Task**

Your task is to create a Python script using BeautifulSoup and pandas to scrape Books to Scrape book data, focusing on book ratings and genres. The script should be able to filter books with ratings above a certain threshold and in specific genres. Additionally, the script should structure the scraped data in a tabular format using pandas for further analysis.

**Expected Outcome**

A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`. The function should scrape book data from the "Books to Scrape" website and return a `pandas` DataFrame with the following columns:

**Expected Outcome**

- A function named `scrape_books` that takes two parameters: `min_rating` and `max_price`.
- The function should return a DataFrame with the following columns:
  - **UPC**: The Universal Product Code (UPC) of the book.
  - **Title**: The title of the book.
  - **Price (£)**: The price of the book in pounds.
  - **Rating**: The rating of the book (1-5 stars).
  - **Genre**: The genre of the book.
  - **Availability**: Whether the book is in stock or not.
  - **Description**: A brief description or product description of the book (if available).
  
You will execute this script to scrape data for books with a minimum rating of `4.0 and above` and a maximum price of `£20`. 

Remember to experiment with different ratings and prices to ensure your code is versatile and can handle various searches effectively!

**Resources**

- [Beautiful Soup Documentation](https://www.crummy.com/software/BeautifulSoup/bs4/doc/)
- [Pandas Documentation](https://pandas.pydata.org/pandas-docs/stable/index.html)
- [Books to Scrape](https://books.toscrape.com/)


**Hint**

Your first mission is to familiarize yourself with the **Books to Scrape** website. Navigate to [Books to Scrape](http://books.toscrape.com/) and explore the available books to understand their layout and structure. 

Next, think about how you can set parameters for your data extraction:

- **Minimum Rating**: Focus on books with a rating of 4.0 and above.
- **Maximum Price**: Filter for books priced up to £20.

After reviewing the site, you can construct a plan for scraping relevant data. Pay attention to the details displayed for each book, including the title, price, rating, and availability. This will help you identify the correct HTML elements to target with your scraping script.

Make sure to build your scraping URL and logic based on the patterns you observe in the HTML structure of the book listings!


---

**Best of luck! Immerse yourself in the world of books, and may the data be with you!**

**Important Note**:

In the fast-changing online world, websites often update and change their structures. When you try this lab, the **Books to Scrape** website might differ from what you expect.

If you encounter issues due to these changes, like new rules or obstacles preventing data extraction, don’t worry! Get creative.

You can choose another website that interests you and is suitable for scraping data. Options like Wikipedia, The New York Times, or even library databases are great alternatives. The main goal remains the same: extract useful data and enhance your web scraping skills while exploring a source of information you enjoy. This is your opportunity to practice and adapt to different web environments!

In [11]:
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin

START_URL = "https://books.toscrape.com/catalogue/page-1.html"

RATING_VALUES = {"One": 1, "Two": 2, "Three": 3, "Four": 4, "Five": 5}


def scrape_books(min_rating, max_price):
    """Scrape Books to Scrape and return books with
    rating >= min_rating and price <= max_price."""
    records = []
    session = requests.Session()
    page_url = START_URL

    while page_url:
        response = session.get(page_url, timeout=15)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, "html.parser")  # bytes -> correct UTF-8

        for book in soup.select("article.product_pod"):
            # Price (on the listing page)
            price_text = book.select_one(".price_color").get_text(strip=True)
            price = float(re.sub(r"[^\d.]", "", price_text))

            # Rating (on the listing page)
            rating_class = next(
                (c for c in book.select_one(".star-rating")["class"] if c != "star-rating"),
                None,
            )
            rating = RATING_VALUES.get(rating_class)

            # Filter first, so detail pages are only fetched for matching books
            if rating is None or rating < min_rating or price > max_price:
                continue

            title = book.h3.a["title"]
            availability = book.select_one(".availability").get_text(" ", strip=True)

            # Detail page: UPC, genre, description
            detail_url = urljoin(page_url, book.h3.a["href"])
            detail_response = session.get(detail_url, timeout=15)
            detail_response.raise_for_status()
            detail_soup = BeautifulSoup(detail_response.content, "html.parser")

            upc_row = next(
                (row for row in detail_soup.select("table tr")
                 if row.th and row.th.get_text(strip=True) == "UPC"),
                None,
            )
            upc = upc_row.td.get_text(strip=True) if upc_row else None

            category_links = detail_soup.select(".breadcrumb li a")
            genre = category_links[-1].get_text(strip=True) if category_links else None

            heading = detail_soup.find(id="product_description")
            desc_tag = heading.find_next("p") if heading else None
            description = desc_tag.get_text(strip=True) if desc_tag else None

            records.append({
                "UPC": upc,
                "Title": title,
                "Price (£)": price,
                "Rating": rating,
                "Genre": genre,
                "Availability": availability,
                "Description": description,
            })
            time.sleep(0.2)   # be polite to the site

        # Go to the next page, or stop when there is no "next" button
        next_link = soup.select_one("li.next a")
        page_url = urljoin(page_url, next_link["href"]) if next_link else None

    return pd.DataFrame(
        records,
        columns=["UPC", "Title", "Price (£)", "Rating", "Genre", "Availability", "Description"],
    )

In [13]:
df = scrape_books(min_rating=4, max_price=20)
print(df.shape)

df.to_csv("books.csv", index=False, encoding="utf-8")

(75, 7)


In [14]:
(df.head())

,UPC,Title,Price (£),Rating,Genre,Availability,Description
0,ce6396b0f23f6ecc,Set Me Free,17.46,5,Young Adult,In stock,Aaron Ledbetter’s future had been planned out ...
1,6258a1f6a6dcfe50,The Four Agreements: A Practical Guide to Pers...,17.66,5,Spirituality,In stock,"In The Four Agreements, don Miguel Ruiz reveal..."
2,6be3beb0793a53e7,Sophie's World,15.94,5,Philosophy,In stock,A page-turning novel that is also an explorati...
3,657fe5ead67a7767,Untitled Collection: Sabbath Poems 2014,14.27,4,Poetry,In stock,"More than thirty-five years ago, when the weat..."
4,51653ef291ab7ddc,This One Summer,19.49,4,Sequential Art,In stock,"Every summer, Rose goes with her mom and dad t..."
